# GloVe — Global Vectors for Word Representation (2014)

_Papers · Sequence & NLP_

**Learn word vectors by fitting their dot products to the log of how often words co-occur across the whole corpus.**

---

This notebook is a practice scaffold for the **GloVe — Global Vectors for Word Representation (2014)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — NumPy

In [ ]:
import numpy as np
import math

# ---- recompute the WORKED EXAMPLE: one term of J (Eq. 8) ----
def f(x, xmax, alpha):                       # Eq. 9 weighting function
    return (x / xmax) ** alpha if x < xmax else 1.0
wi  = np.array([0.5, -0.2, 0.1]); wj = np.array([0.3, 0.4, -0.1])
bi, bj, X_ij = 0.2, 0.1, 30.0
w  = f(X_ij, 100.0, 0.75)                     # 0.405360
dot = wi @ wj                                 # 0.06
res = dot + bi + bj - math.log(X_ij)          # -3.041197
term = w * res ** 2                           # 3.749127
print("f(30)      :", round(w, 6))            # 0.40536
print("dot        :", round(dot, 6))          # 0.06
print("log X      :", round(math.log(X_ij), 6))  # 3.401197
print("residual   :", round(res, 6))          # -3.041197
print("term of J  :", round(term, 6))         # 3.749127

# ---- build the GLOBAL co-occurrence matrix X from a tiny two-topic corpus ----
roy = ['king','queen','prince','princess','man','woman','boy','girl']
ani = ['cat','dog','kitten','puppy']
rng = np.random.default_rng(42)
sents  = [list(rng.choice(roy, size=5, replace=True)) for _ in range(40)]
sents += [list(rng.choice(ani, size=4, replace=True)) for _ in range(30)]
vocab = sorted(set(w for s in sents for w in s)); stoi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
X = np.zeros((V, V))
for s in sents:                               # ONE global matrix, not per-window training
    for a in range(len(s)):
        for b in range(len(s)):
            if a != b:
                X[stoi[s[a]], stoi[s[b]]] += 1.0 / abs(a - b)   # 1/distance weighting
print("V =", V, " nonzero entries =", int((X > 0).sum()))

# ---- minimize the GloVe objective J (Eq. 8) with explicit Adagrad ----
D, xmax, alpha, lr = 16, 30.0, 0.75, 0.05
r = np.random.default_rng(7)
W  = r.standard_normal((V, D)) * 0.1; Wt = r.standard_normal((V, D)) * 0.1
b  = np.zeros(V); bt = np.zeros(V)
gW = np.ones_like(W); gWt = np.ones_like(Wt); gb = np.ones_like(b); gbt = np.ones_like(bt)
nz = np.argwhere(X > 0)                        # only the populated cells contribute
def fw(x): return np.where(x < xmax, (x / xmax) ** alpha, 1.0)
for epoch in range(250):
    r.shuffle(nz); J = 0.0
    for i, j in nz:
        xij = X[i, j]
        res = W[i] @ Wt[j] + b[i] + bt[j] - np.log(xij)   # Eq. 7 residual
        wt  = fw(xij)                                       # Eq. 9 weight
        J  += wt * res ** 2                                 # Eq. 8 term
        g   = 2 * wt * res                                  # dJ/d(residual)
        gwi, gwj = g * Wt[j], g * W[i]
        W[i]  -= lr / np.sqrt(gW[i])  * gwi; gW[i]  += gwi ** 2   # Adagrad
        Wt[j] -= lr / np.sqrt(gWt[j]) * gwj; gWt[j] += gwj ** 2
        b[i]  -= lr / np.sqrt(gb[i])  * g;   gb[i]  += g ** 2
        bt[j] -= lr / np.sqrt(gbt[j]) * g;   gbt[j] += g ** 2
print("final J    :", round(J, 4))             # ~0.0016

# ---- keep w + w_tilde as the embeddings; nearest neighbors + analogy ----
E  = W + Wt
En = E / np.linalg.norm(E, axis=1, keepdims=True)
sim = En @ En.T
def neighbors(word, k=3):
    i = stoi[word]; s = sim[i].copy(); s[i] = -9
    return [vocab[t] for t in np.argsort(-s)[:k]]
for word in ['king', 'cat', 'man', 'dog']:
    print(f"nearest to {word:5s}:", neighbors(word))
# -> cat: ['kitten','dog','puppy']   king: ['man','boy','queen']

v = En[stoi['king']] - En[stoi['man']] + En[stoi['woman']]   # the analogy direction
v = v / np.linalg.norm(v)
s = En @ v
for w in ['king', 'man', 'woman']: s[stoi[w]] = -9            # exclude the inputs
print("king - man + woman ->", [vocab[t] for t in np.argsort(-s)[:3]])
# -> ['queen', 'boy', 'prince']

## Visualize the data & results

_Fit GloVe to a global co-occurrence matrix built from a tiny corpus that mixes royalty/people words and animal words — do the learned vectors cluster by topic (and support the king-man+woman analogy) without ever being told the topics?_

In [ ]:
import numpy as np
roy = ['king','queen','prince','princess','man','woman','boy','girl']
ani = ['cat','dog','kitten','puppy']
rng = np.random.default_rng(42)
sents  = [list(rng.choice(roy, size=5, replace=True)) for _ in range(40)]
sents += [list(rng.choice(ani, size=4, replace=True)) for _ in range(30)]
vocab = sorted(set(w for s in sents for w in s)); stoi = {w: i for i, w in enumerate(vocab)}; V = len(vocab)
X = np.zeros((V, V))
for s in sents:
    for a in range(len(s)):
        for b in range(len(s)):
            if a != b: X[stoi[s[a]], stoi[s[b]]] += 1.0 / abs(a - b)

D, xmax, alpha, lr = 16, 30.0, 0.75, 0.05
r = np.random.default_rng(7)
W  = r.standard_normal((V, D)) * 0.1; Wt = r.standard_normal((V, D)) * 0.1
b  = np.zeros(V); bt = np.zeros(V)
gW = np.ones_like(W); gWt = np.ones_like(Wt); gb = np.ones_like(b); gbt = np.ones_like(bt)
nz = np.argwhere(X > 0)
fw = lambda x: np.where(x < xmax, (x / xmax) ** alpha, 1.0)
for epoch in range(250):
    r.shuffle(nz)
    for i, j in nz:
        xij = X[i, j]
        res = W[i] @ Wt[j] + b[i] + bt[j] - np.log(xij); wt = fw(xij); g = 2 * wt * res
        gwi, gwj = g * Wt[j], g * W[i]
        W[i]  -= lr / np.sqrt(gW[i])  * gwi; gW[i]  += gwi ** 2
        Wt[j] -= lr / np.sqrt(gWt[j]) * gwj; gWt[j] += gwj ** 2
        b[i]  -= lr / np.sqrt(gb[i])  * g;   gb[i]  += g ** 2
        bt[j] -= lr / np.sqrt(gbt[j]) * g;   gbt[j] += g ** 2

E  = W + Wt                                    # GloVe sums the two vector sets
Ec = E - E.mean(0)                             # center, then PCA via SVD
U, S, Vt = np.linalg.svd(Ec, full_matrices=False)
P  = Ec @ Vt[:2].T
P  = P / np.abs(P).max() * 2.0                 # scale to a readable range
for w in vocab:
    x, y = P[stoi[w]]
    print(f"{w:9s} {x:6.3f} {y:6.3f}")          # the labeled coordinates plotted above

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute one term of $J$ for a pair with $X_{ij}=50$, $x_{\max}=100$, $\alpha=0.75$, where $w_i^\top\tilde w_j = 2.5$, $b_i=0.5$, $\tilde b_j=0.5$. (Use $\ln 50 = 3.912023$.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Weight: $f(50)=(50/100)^{0.75}=0.5^{0.75}=0.594604$. — _$50\lt x_{\max}$, so use the power-law branch of Eq. 9._
- Residual: $2.5+0.5+0.5-3.912023=-0.412023$. — _Model prediction minus the target $\log X_{ij}$ (Eq. 8 inner term)._
- Square: $(-0.412023)^2=0.169763$. — _Least-squares penalizes the error symmetrically._
- Term: $0.594604\times0.169763=0.100942$. — _Scale the squared error by the weight._

**Answer:** The term is $\approx 0.1009$. Because the model's prediction $3.5$ is already close to $\log 50\approx 3.912$, the residual is small and this pair contributes little to $J$.

</details>

**Problem 2.** Using Table 1's logic, the corpus gives $P(\text{solid}\mid\text{ice})=1.9\times10^{-4}$ and $P(\text{solid}\mid\text{steam})=2.2\times10^{-5}$. Why does GloVe build its model around the ratio of these rather than either probability alone?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ratio $=1.9\times10^{-4} / 2.2\times10^{-5}\approx 8.6$ (paper rounds to 8.9). — _A ratio much greater than 1 flags that 'solid' is specific to 'ice'._
- Compare to an irrelevant probe like 'water': its ratio is $\approx 1.36$, near 1. — _Words related to both (or neither) cancel in the ratio._
- Note both raw probabilities are tiny ($\sim10^{-4}$) and hard to compare directly. — _Raw magnitudes are dominated by overall word frequency, not relatedness._

**Answer:** The ratio cancels out frequency effects and non-discriminative context words (their ratio $\approx 1$), leaving only the signal that distinguishes 'ice' from 'steam'. That is why Eq. 1 sets the model's target to $P_{ik}/P_{jk}$, which after the homomorphism argument becomes the $\log X_{ij}$ regression of Eq. 7.

</details>

**Problem 3.** Ablation: in the CODE, replace the weighting $f(X_{ij})$ with the constant $1$ (plain, unweighted least squares on $\log X$). What should you expect to change, and what should stay the same?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set f to return 1.0 for every count and retrain on the same $X$. — _Removes GloVe's down-weighting of rare and very frequent pairs (Eq. 9 → constant)._
- Watch the rare/noisy pairs (small $X_{ij}$): they now contribute as much as frequent ones. — _Property 2 of $f$ (non-decreasing, small for rare) is what protected against this._
- Re-check the topic clustering and the analogy. — _See whether the qualitative effect survives the change._

**Answer:** With $f\equiv 1$, every nonzero pair counts equally, so a few very frequent pairs and many noisy rare pairs both pull harder on the fit; on a real corpus this degrades the vectors (the paper reports the weighted version helps). On our tiny clean toy corpus the clustering and analogy usually still appear, but the loss landscape is worse-conditioned and the result is more seed-sensitive — illustrating exactly why GloVe introduced $f$.

</details>